<a href="https://colab.research.google.com/github/JF-74/Laboratorio-Implementaci-n-de-Redes-Neuronales-MLP-MLflow-/blob/main/Laboratorio_Implementaci%C3%B3n_de_Redes_Neuronales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**nombre**: Jose aravena

**Asignatura**: mineria de datos

**numero de lista**: 3

In [1]:
!pip install -q mlflow pyngrok

In [2]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, mlflow

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             ConfusionMatrixDisplay)

In [3]:
SEED = 3
random.seed(SEED)
np.random.seed(SEED)

print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("mlflow      :", mlflow.__version__)
print("SEED        :", SEED)

pandas      : 2.2.3
scikit-learn: 1.6.1
mlflow      : 3.18.0
SEED        : 3


In [4]:
import kagglehub
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

ruta = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print("Carpeta:", ruta)
print("Archivos:", os.listdir(ruta))

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Carpeta: /kaggle/input/breast-cancer-wisconsin-data
Archivos: ['data.csv']


In [5]:
csv_path = os.path.join(ruta, "data.csv")
df = pd.read_csv(csv_path)
print(df.shape)
df.info()

(569, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se    

In [6]:
nulos = df.isna().sum()
print(nulos[nulos > 0])

df = df.drop(columns=["Unnamed: 32", "id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

print(df.shape)
print(df["diagnosis"].value_counts())
print(df["diagnosis"].value_counts(normalize=True).round(3))

Unnamed: 32    569
dtype: int64
(569, 31)
diagnosis
0    357
1    212
Name: count, dtype: int64
diagnosis
0    0.627
1    0.373
Name: proportion, dtype: float64


In [7]:
cols = ["radius_mean", "area_mean", "smoothness_mean"]
resumen = df[cols].agg(["min", "max"]).T
resumen["rango"] = resumen["max"] - resumen["min"]
print(resumen)

r = resumen["rango"]
print(f"\nrango(area_mean) / rango(radius_mean)     = {r['area_mean']/r['radius_mean']:.1f}")
print(f"rango(area_mean) / rango(smoothness_mean) = {r['area_mean']/r['smoothness_mean']:.1f}")

                       min        max       rango
radius_mean        6.98100    28.1100    21.12900
area_mean        143.50000  2501.0000  2357.50000
smoothness_mean    0.05263     0.1634     0.11077

rango(area_mean) / rango(radius_mean)     = 111.6
rango(area_mean) / rango(smoothness_mean) = 21282.8


In [8]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Proporción clase maligna -> train: %.3f | test: %.3f | total: %.3f"
      % (y_train.mean(), y_test.mean(), y.mean()))

Train: (455, 30)  Test: (114, 30)
Proporción clase maligna -> train: 0.374 | test: 0.368 | total: 0.373


In [9]:
import os, subprocess, time, requests
from google.colab import userdata
from pyngrok import ngrok

def servidor_activo():
    try:
        return requests.get("http://127.0.0.1:5000/health", timeout=1).ok
    except requests.RequestException:
        return False

# 1) Servidor MLflow: solo se lanza si no está ya corriendo
if not servidor_activo():
    env = dict(os.environ,
               MLFLOW_SERVER_ALLOWED_HOSTS="*",
               MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")
    subprocess.Popen(["mlflow", "server",
                      "--backend-store-uri", "sqlite:///mlflow.db",
                      "--host", "127.0.0.1", "--port", "5000"], env=env)

    for _ in range(60):
        if servidor_activo():
            break
        time.sleep(1)

print("Servidor MLflow activo:", servidor_activo())

# 2) Authtoken: si ya estaba guardado, se ignora el error
try:
    ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
except Exception as e:
    print("Authtoken ya configurado, se continúa:", str(e)[:80])

# 3) Túnel público
ngrok.kill()
tunel = ngrok.connect(5000)
print("MLflow UI:", tunel.public_url)

Servidor MLflow activo: False
Authtoken ya configurado, se continúa: An error occurred when saving the auth token: b'Authtoken already set in configu
MLflow UI: https://survive-chirping-easter.ngrok-free.dev


In [10]:
print(ngrok.get_tunnels())

[<NgrokTunnel: "https://survive-chirping-easter.ngrok-free.dev" -> "http://localhost:5000">]


In [11]:
APELLIDO = "aravena"
EXPERIMENTO = f"IEI097-L2.12-{APELLIDO}-v2"

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENTO)
print("Experimento:", EXPERIMENTO)

2026/11/11 02:07:18 INFO mlflow.tracking.fluent: Experiment with name 'IEI097-L2.12-aravena-v2' does not exist. Creating a new experiment.


Experimento: IEI097-L2.12-aravena-v2


In [12]:
def correr_experimento(nombre, escalar, capas, activacion, fase,
                       seed_modelo=SEED, alpha=1e-4, max_iter=500,
                       lr_init=1e-3):
    pasos = []
    if escalar:
        pasos.append(("scaler", StandardScaler()))
    pasos.append(("mlp", MLPClassifier(
        hidden_layer_sizes=capas, activation=activacion, alpha=alpha,
        max_iter=max_iter, learning_rate_init=lr_init,
        random_state=seed_modelo)))
    pipe = Pipeline(pasos)

    with mlflow.start_run(run_name=nombre):
        mlflow.set_tag("fase", fase)

        mlflow.log_params({
            "arquitectura": str(capas),
            "activacion": activacion,
            "alpha": alpha,
            "max_iter": max_iter,
            "learning_rate_init": lr_init,
            "random_state": seed_modelo,
            "escalado": escalar,
            "semilla_particion": SEED,
        })

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pipe.fit(X_train, y_train)

        mlp = pipe.named_steps["mlp"]
        pred_tr = pipe.predict(X_train)
        pred_te = pipe.predict(X_test)

        metricas = {
            "acc_train": accuracy_score(y_train, pred_tr),
            "acc_test": accuracy_score(y_test, pred_te),
            "recall_maligno": recall_score(y_test, pred_te, pos_label=1),
            "f1_maligno": f1_score(y_test, pred_te, pos_label=1),
            "n_iter": mlp.n_iter_,
        }
        mlflow.log_metrics(metricas)

        for epoca, perdida in enumerate(mlp.loss_curve_):
            mlflow.log_metric("train_loss", perdida, step=epoca)

        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_test, pred_te, display_labels=["Benigno", "Maligno"], ax=ax)
        ax.set_title(nombre)
        mlflow.log_figure(fig, "matriz_confusion.png")
        plt.close(fig)

    print(f"{nombre:<14} acc_test={metricas['acc_test']:.4f} "
          f"recall_M={metricas['recall_maligno']:.4f} "
          f"f1={metricas['f1_maligno']:.4f} iter={metricas['n_iter']}")
    return metricas

In [13]:
# Run A
mA = correr_experimento("A_sin_escalar_relu_30", escalar=False,
                        capas=(30,), activacion="relu", fase="experimentos")

🏃 View run A_sin_escalar_relu_30 at: http://127.0.0.1:5000/#/experiments/2/runs/e794e524fa65470386657b4cb316dfa6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
A_sin_escalar_relu_30 acc_test=0.9561 recall_M=0.9286 f1=0.9398 iter=170


In [14]:
# Run B
mB = correr_experimento("B_escalado_relu_30", escalar=True,
                        capas=(30,), activacion="relu", fase="experimentos")

🏃 View run B_escalado_relu_30 at: http://127.0.0.1:5000/#/experiments/2/runs/04dbd66a24554c4db7a483dd727f74a7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
B_escalado_relu_30 acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=396


In [15]:
# Run C
mC = correr_experimento("C_escalado_relu_64_32", escalar=True,
                        capas=(64, 32), activacion="relu", fase="experimentos")

🏃 View run C_escalado_relu_64_32 at: http://127.0.0.1:5000/#/experiments/2/runs/e5a56458456a40b59324364068be0a3b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
C_escalado_relu_64_32 acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=228


In [16]:
# Run D
mD = correr_experimento("D_escalado_tanh_64_32", escalar=True,
                        capas=(64, 32), activacion="tanh", fase="experimentos")

🏃 View run D_escalado_tanh_64_32 at: http://127.0.0.1:5000/#/experiments/2/runs/076562b52d6849368c3add76130a737f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
D_escalado_tanh_64_32 acc_test=1.0000 recall_M=1.0000 f1=1.0000 iter=308


In [17]:
resumen_exp = pd.DataFrame({"A": mA, "B": mB, "C": mC, "D": mD}).T
print(resumen_exp.round(4))

print("\nEfecto del escalado (B - A):")
print((pd.Series(mB) - pd.Series(mA)).round(4))

   acc_train  acc_test  recall_maligno  f1_maligno  n_iter
A     0.9319    0.9561          0.9286      0.9398   170.0
B     0.9912    0.9912          0.9762      0.9880   396.0
C     1.0000    0.9912          0.9762      0.9880   228.0
D     1.0000    1.0000          1.0000      1.0000   308.0

Efecto del escalado (B - A):
acc_train           0.0593
acc_test            0.0351
recall_maligno      0.0476
f1_maligno          0.0482
n_iter            226.0000
dtype: float64


In [18]:
mE = correr_experimento("E_alpha_0.01", escalar=True, capas=(64, 32),
                        activacion="relu", alpha=1e-2, fase="experimentos")
mF = correr_experimento("F_lr_0.01", escalar=True, capas=(64, 32),
                        activacion="relu", lr_init=1e-2, fase="experimentos")

🏃 View run E_alpha_0.01 at: http://127.0.0.1:5000/#/experiments/2/runs/909092b624b1498497e55c6a469d85c4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
E_alpha_0.01   acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=228
🏃 View run F_lr_0.01 at: http://127.0.0.1:5000/#/experiments/2/runs/5fbb1c894fc94abd9e82980aaf7720eb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
F_lr_0.01      acc_test=0.9825 recall_M=0.9762 f1=0.9762 iter=58


In [19]:
res_semillas = []
for s in range(SEED, SEED + 5):
    m = correr_experimento(f"C_seed{s}", escalar=True, capas=(64, 32),
                           activacion="relu", fase="semillas", seed_modelo=s)
    res_semillas.append(m)

🏃 View run C_seed3 at: http://127.0.0.1:5000/#/experiments/2/runs/962262ea62ab4d7ea2733151b91b95ed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
C_seed3        acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=228
🏃 View run C_seed4 at: http://127.0.0.1:5000/#/experiments/2/runs/fffb0928c15a47da8ec8e46891ba9c3b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
C_seed4        acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=212
🏃 View run C_seed5 at: http://127.0.0.1:5000/#/experiments/2/runs/98a5f9e7c38648c5b7bad87c3bc6a27a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
C_seed5        acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=205
🏃 View run C_seed6 at: http://127.0.0.1:5000/#/experiments/2/runs/56635e0c9dd148cb8757562ca5974e3f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
C_seed6        acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=175
🏃 View run C_seed7 at: http://127.0.0.1:5000/#/experiments/2/runs/77add62dbd0647c89ce8307b537a9e47
🧪

In [20]:
m_rep = correr_experimento("C_replica", escalar=True, capas=(64, 32),
                           activacion="relu", fase="replica", seed_modelo=SEED)

print("\nComparación run C vs réplica:")
for k in ["acc_train", "acc_test", "recall_maligno", "f1_maligno", "n_iter"]:
    print(f"  {k:<15} C={mC[k]:.12f}  replica={m_rep[k]:.12f}  "
          f"dif={abs(mC[k]-m_rep[k]):.2e}")

🏃 View run C_replica at: http://127.0.0.1:5000/#/experiments/2/runs/7c33f84cd9b348649abb9575af9f3796
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
C_replica      acc_test=0.9912 recall_M=0.9762 f1=0.9880 iter=228

Comparación run C vs réplica:
  acc_train       C=1.000000000000  replica=1.000000000000  dif=0.00e+00
  acc_test        C=0.991228070175  replica=0.991228070175  dif=0.00e+00
  recall_maligno  C=0.976190476190  replica=0.976190476190  dif=0.00e+00
  f1_maligno      C=0.987951807229  replica=0.987951807229  dif=0.00e+00
  n_iter          C=228.000000000000  replica=228.000000000000  dif=0.00e+00


In [21]:
df_sem = pd.DataFrame(res_semillas)
tabla = df_sem[["acc_test", "recall_maligno"]].agg(["min", "max"]).T
tabla["rango"] = tabla["max"] - tabla["min"]
print(tabla.round(4))

                   min     max  rango
acc_test        0.9912  0.9912    0.0
recall_maligno  0.9762  0.9762    0.0


In [22]:
rango_sem_acc = df_sem["acc_test"].max() - df_sem["acc_test"].min()
rango_sem_rec = df_sem["recall_maligno"].max() - df_sem["recall_maligno"].min()
cfg = resumen_exp.loc[["B", "C", "D"]]
rango_cfg_acc = cfg["acc_test"].max() - cfg["acc_test"].min()
rango_cfg_rec = cfg["recall_maligno"].max() - cfg["recall_maligno"].min()

print(f"Entre semillas (config C):   acc_test={rango_sem_acc:.4f}  recall={rango_sem_rec:.4f}")
print(f"Entre configuraciones B,C,D: acc_test={rango_cfg_acc:.4f}  recall={rango_cfg_rec:.4f}")

Entre semillas (config C):   acc_test=0.0000  recall=0.0000
Entre configuraciones B,C,D: acc_test=0.0088  recall=0.0238


In [23]:
_r = mlflow.search_runs(experiment_names=[EXPERIMENTO])
print(f"Runs registrados: {len(_r)} (mínimo 10) ->", "OK" if len(_r) >= 10 else "FALTAN")
print(_r["tags.fase"].value_counts(), "\n")
_n = _r[_r["tags.fase"] == "semillas"]["params.random_state"].nunique()
print(f"Semillas distintas: {_n} (deben ser 5) ->", "OK" if _n == 5 else "REVISAR")
_d = max(abs(mC[k] - m_rep[k]) for k in mC)
print(f"Máx. diferencia run C vs réplica: {_d:.2e} ->", "OK" if _d < 1e-9 else "REVISAR")

Runs registrados: 12 (mínimo 10) -> OK
tags.fase
experimentos    6
semillas        5
replica         1
Name: count, dtype: int64 

Semillas distintas: 5 (deben ser 5) -> OK
Máx. diferencia run C vs réplica: 0.00e+00 -> OK


In [24]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
print("Total de runs:", len(runs))

cols = ["tags.mlflow.runName", "tags.fase",
        "params.arquitectura", "params.activacion", "params.alpha",
        "params.escalado", "params.random_state", "params.semilla_particion",
        "metrics.acc_train", "metrics.acc_test", "metrics.recall_maligno",
        "metrics.f1_maligno", "metrics.n_iter"]
cols = [c for c in cols if c in runs.columns]
runs_out = runs[cols].sort_values("metrics.recall_maligno", ascending=False)
runs_out.to_csv("runs.csv", index=False)
runs_out

Total de runs: 12


,tags.mlflow.runName,tags.fase,params.arquitectura,params.activacion,params.alpha,params.escalado,params.random_state,params.semilla_particion,metrics.acc_train,metrics.acc_test,metrics.recall_maligno,metrics.f1_maligno,metrics.n_iter
8,D_escalado_tanh_64_32,experimentos,"(64, 32)",tanh,0.0001,True,3,3,1.000000,1.000000,1.000000,1.000000,308.0
0,C_replica,replica,"(64, 32)",relu,0.0001,True,3,3,1.000000,0.991228,0.976190,0.987952,228.0
2,C_seed6,semillas,"(64, 32)",relu,0.0001,True,6,3,1.000000,0.991228,0.976190,0.987952,175.0
1,C_seed7,semillas,"(64, 32)",relu,0.0001,True,7,3,1.000000,0.991228,0.976190,0.987952,181.0
3,C_seed5,semillas,"(64, 32)",relu,0.0001,True,5,3,1.000000,0.991228,0.976190,0.987952,205.0
4,C_seed4,semillas,"(64, 32)",relu,0.0001,True,4,3,1.000000,0.991228,0.976190,0.987952,212.0
6,F_lr_0.01,experimentos,"(64, 32)",relu,0.0001,True,3,3,1.000000,0.982456,0.976190,0.976190,58.0
5,C_seed3,semillas,"(64, 32)",relu,0.0001,True,3,3,1.000000,0.991228,0.976190,0.987952,228.0
7,E_alpha_0.01,experimentos,"(64, 32)",relu,0.01,True,3,3,1.000000,0.991228,0.976190,0.987952,228.0
9,C_escalado_relu_64_32,experimentos,"(64, 32)",relu,0.0001,True,3,3,1.000000,0.991228,0.976190,0.987952,228.0


In [25]:
from google.colab import files
files.download("runs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
ngrok.kill()